In [ ]:
"""
Visualization functions
"""

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
from matplotlib.colors import LogNorm


def plot_state_distribution(df_counts: pd.DataFrame):
    """Plot state distribution with frequency weighting."""
    plt.figure(figsize=(8, 6), dpi=150)
    
    # Reference lines
    plt.axhline(2, color='gray', linestyle='--', alpha=0.5)
    plt.axvline(2, color='gray', linestyle='--', alpha=0.5)
    
    # Label regions
    plt.text(1, 4.5, 'Bleeding-Hypocoagulable', fontsize=10, color='#FF6666', alpha=0.9)
    plt.text(4, 4.5, 'Bleeding-Hypercoagulable', fontsize=10, color='#FF9966', alpha=0.9)
    plt.text(1, 1, 'Thrombosis-Hypocoagulable', fontsize=10, color='#6699FF', alpha=0.9)
    plt.text(4, 1, 'Thrombosis-Hypercoagulable', fontsize=10, color='#9933CC', alpha=0.9)
    plt.text(0, 0, 'Normal', fontsize=10, color='#33CC33', alpha=0.9, ha='center')
    
    # Frequency-weighted scatter plot
    sc = plt.scatter(
        df_counts["Y_x"], 
        df_counts["Y_y"], 
        s=df_counts["count"] * 0.5,
        c=df_counts["count"],
        cmap="viridis", 
        alpha=0.7, 
        edgecolor="black", 
        linewidth=0.3
    )
    
    plt.colorbar(sc, label="Frequency")
    plt.xlabel('Thrombosis tendency (x)')
    plt.ylabel('Bleeding tendency (y)')
    plt.xlim(-0.5, 6.5)
    plt.ylim(-0.5, 7.5)
    plt.tight_layout()
    
    return plt.gcf()


def plot_treatment_distribution(df_dynamic: pd.DataFrame, jitter_strength: float = 0.15):
    """Plot treatment distribution with jitter."""
    plt.figure(figsize=(8, 6))
    
    # Get all treatment categories
    treat_groups = df_dynamic['Treat_group'].dropna().unique()
    colors = plt.cm.tab10(np.linspace(0, 1, len(treat_groups)))
    markers = ['o', 's', 'v', '*', 'H', '+']
    
    for tg, color, marker in zip(treat_groups, colors, markers):
        df_tg = df_dynamic[df_dynamic['Treat_group'] == tg]
        df_counts = df_tg.groupby(['Y_x', 'Y_y']).size().reset_index(name='count')
        
        # Add jitter
        x_jitter = df_counts['Y_x'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(df_counts))
        y_jitter = df_counts['Y_y'] + np.random.uniform(-jitter_strength, jitter_strength, size=len(df_counts))
        
        plt.scatter(
            x_jitter,
            y_jitter,
            s=np.sqrt(df_counts['count']) * 5,
            color=color,
            marker=marker,
            alpha=0.7,
            edgecolor='black',
            linewidth=0.3,
            label=tg
        )
    
    plt.xlabel('Coagulation Tendency (x)')
    plt.ylabel('Bleeding Tendency (y)')
    plt.xlim(-0.2, 5.2)
    plt.ylim(-0.2, 5.8)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    
    return plt.gcf()


def plot_reward_distribution(df_rl: pd.DataFrame):
    """Plot reward distribution violin plot."""
    plt.figure(figsize=(8, 6), dpi=200)
    
    df_plot = df_rl[['ID', 'reward_smooth', 'reward_shaped']].fillna(0)
    df_melt = df_plot.melt(
        id_vars='ID', 
        value_vars=['reward_smooth', 'reward_shaped'],
        var_name='Reward Type', 
        value_name='Value'
    )
    
    # Rename for better display
    name_map = {
        'reward_smooth': 'Smoothed reward',
        'reward_shaped': 'Shaped reward'
    }
    df_melt['Reward Type'] = df_melt['Reward Type'].map(name_map)
    
    # Create violin plot
    sns.violinplot(
        x="Reward Type", 
        y="Value", 
        data=df_melt, 
        palette="Set2", 
        inner="box", 
        saturation=1,  
        linewidth=1.2  
    )
    
    plt.xlabel("")
    plt.ylabel("Reward value")
    sns.despine()
    plt.tight_layout()
    
    return plt.gcf()


def plot_training_results(result, per_patient_df: pd.DataFrame):
    """Plot training results and per-patient rewards."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=False)
    
    # Training/Test Loss
    ax1.plot(result.train_loss_list, label='Train Loss', color='#1f77b4')
    ax1.plot(result.test_loss_list, '--', label='Test Loss', color='#ff7f0e')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.set_title('Training/Test Loss')
    
    # Per-patient Reward
    num_patients = len(per_patient_df)
    per_patient_df = per_patient_df.sort_values('avg_reward', ascending=False).reset_index(drop=True)
    
    # Calculate error
    avg_reward = per_patient_df['avg_reward'].mean()
    std_reward = per_patient_df['avg_reward'].std()
    yerr = std_reward / np.sqrt(num_patients)
    
    ax2.errorbar(
        np.arange(num_patients),
        per_patient_df['avg_reward'],
        yerr=yerr,
        fmt='o',
        ecolor='gray',
        capsize=4,
        label='Per-patient reward'
    )
    
    # Highlight top and bottom
    top5_idx = per_patient_df.index[:5]
    bottom5_idx = per_patient_df.index[-5:]
    
    for i in range(num_patients):
        if i in top5_idx:
            ax2.plot(i, per_patient_df['avg_reward'][i], 'o', color='green', markersize=8)
        elif i in bottom5_idx:
            ax2.plot(i, per_patient_df['avg_reward'][i], 'o', color='red', markersize=8)
    
    ax2.axhline(avg_reward, color='magenta', linestyle='--', label='Average reward')
    ax2.set_xlabel('Patients')
    ax2.set_ylabel('Per-patient reward')
    ax2.legend()
    ax2.set_title('Per-Patient Reward ± 95% CI')
    
    plt.tight_layout()
    return fig